## Conversion of ChEBI IDs

ChEBI entities are sometimes listed with their primary ID, and sometimes with their secondary ID. This notebook aims to create a relationship between all primary and secondary IDs in order to ensure correct mapping throughout the master. Finally, a function for the conversion from secondary to primary (s2p) ID is provided. The file used, is ChEBI_complete_3star.sdf, and the most recent update is 2nd of April 2025.

In [1]:
import pandas as pd
from rdkit import Chem
import numpy as np

In [2]:
def extract_id_mappings(sdf_file):
    supplier = Chem.SDMolSupplier(sdf_file)
    
    data = []
    for mol in supplier:
        if mol is None:
            continue
        
        primary_id = mol.GetProp("ChEBI ID")
        
        if mol.HasProp("Secondary ChEBI ID"):
            secondary_ids = mol.GetProp("Secondary ChEBI ID").split("\n")
        else:
            secondary_ids = []
        
        for sec_id in secondary_ids:
            data.append({"Primary_ID": primary_id, "Secondary_ID": sec_id})
    
    df = pd.DataFrame(data)
    return df

A lot of warnings and errors occur, but the whenever there are primary and secondary ID relations, they are extracted.

In [ ]:
sdf_file = "../files/ChEBI_complete_3star.sdf"
mappings_df = extract_id_mappings(sdf_file)
mappings_df.to_csv("s2p.tsv", sep="\t", index=False)

Following is a function that can be copied and used throughout the project for s2p conversion. This function returns exactly what was given, if it can't be found in the s2p.tsv file. This is because s2p.tsv only contain the ChEBI IDs where there are secondary IDs. And far from all entities have secondary IDs. Else, it always returns the primary ID of the submitted ID.

In [7]:
df = pd.read_csv("s2p.tsv", sep="\t")
secondary_to_primary = dict(zip(df["Secondary_ID"], df["Primary_ID"]))
def s2p(chebi_id):
    return secondary_to_primary.get(chebi_id, chebi_id)

Below is a funciton to extract ChEBI Name from the primary ID.

In [8]:
def extract_id_name_mapping(sdf_file):
    supplier = Chem.SDMolSupplier(sdf_file)
    
    data = []
    for mol in supplier:
        if mol is None:
            continue
        
        primary_id = mol.GetProp("ChEBI ID")
        
        if mol.HasProp("ChEBI Name"):
            chebi_name = mol.GetProp("ChEBI Name")
        else:
            chebi_name = np.nan
        

        data.append({"Primary_ID": primary_id, "ChEBI Name": chebi_name})
    
    df = pd.DataFrame(data)
    return df

In [ ]:
sdf_file = "../files/ChEBI_complete_3star.sdf"
mappings_df = extract_id_name_mapping(sdf_file)
mappings_df.to_csv("p2n.tsv", sep="\t", index=False)

Usage example:

In [10]:
df = pd.read_csv("p2n.tsv", sep="\t")
primary_to_name = dict(zip(df["ChEBI Name"], df["Primary_ID"]))
def p2n(chebi_id):
    return primary_to_name.get(chebi_id, chebi_id)